In [1]:
import json
import pandas as pd
from tqdm import tqdm
import re
import numpy as np
import matplotlib.pyplot as plt


## Drugs @ FDA

In [2]:
fda_drugs_file = "/shares/animalwelfare.crs.uzh/fda_files/2025_11_11/drug-drugsfda-0001-of-0001.json"

In [3]:

# Load the FDA JSON file
with open(fda_drugs_file, "r") as f:
    data = json.load(f)

# Initialize list for filtered results
orig_records = []

# Loop over results
for result in data.get("results", []):
    submissions = result.get("submissions", [])
    
    # Find the ORIG submission(s)
    orig_subs = [s for s in submissions if s.get("submission_type") == "ORIG"]
    if not orig_subs:
        continue

    orig = orig_subs[0]
    submission_date = orig.get("submission_status_date")
    year = submission_date[:4] if submission_date else None
    # Extract base info
    record = {
        "application_number": result.get("application_number"),
        "sponsor_name": result.get("sponsor_name"),
        "submission_type": orig.get("submission_type"),
        "submission_status_date": submission_date,
        "submission_class_code_description": orig.get("submission_class_code_description"),
        "year": year
    }

    # Extract first product if exists
    products = result.get("products", [])
    if products:
        p = products[0]
        record.update({
            "brand_name": p.get("brand_name"),
            "dosage_form": p.get("dosage_form"),
            "route": p.get("route"),
            "marketing_status": p.get("marketing_status"),
            "active_ingredients": ", ".join(
                f"{a.get('name')}" for a in p.get("active_ingredients", [])
            )
        })

    # If openfda section exists, flatten a few key fields
    openfda = result.get("openfda", {})
    if openfda:
        record.update({
            "openfda_brand_name": ", ".join(openfda.get("brand_name", [])),
            "openfda_generic_name": ", ".join(openfda.get("generic_name", [])),
            "openfda_route": ", ".join(openfda.get("route", [])),
            "openfda_substance_name": ", ".join(openfda.get("substance_name", []))
        })

    orig_records.append(record)

# Convert to DataFrame for easier viewing/analysis
df_orig = pd.DataFrame(orig_records)

In [4]:
df_orig.shape

(25712, 15)

In [17]:
# Show summary
print("Found", len(df_orig), "ORIG applications")
df_orig.head()

Found 25712 ORIG applications


,application_number,sponsor_name,submission_type,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,active_ingredients,openfda_brand_name,openfda_generic_name,openfda_route,openfda_substance_name
0,ANDA076194,WATSON LABS,ORIG,20020701,None,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,Prescription,"HYDROCHLOROTHIAZIDE, LISINOPRIL",LISINOPRIL AND HYDROCHLOROTHIAZIDE,LISINOPRIL AND HYDROCHLOROTHIAZIDE,ORAL,"HYDROCHLOROTHIAZIDE, LISINOPRIL"
1,ANDA076206,ROCKWELL MEDCL,ORIG,20030917,None,2003,CALCITRIOL,INJECTABLE,INJECTION,Discontinued,CALCITRIOL,NaN,NaN,NaN,NaN
2,ANDA076212,APOTEX,ORIG,20040616,None,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,Discontinued,"CARBIDOPA, LEVODOPA",NaN,NaN,NaN,NaN
3,ANDA076215,FOUGERA PHARMS,ORIG,20031209,None,2003,BETAMETHASONE DIPROPIONATE,"CREAM, AUGMENTED",TOPICAL,Prescription,BETAMETHASONE DIPROPIONATE,NaN,NaN,NaN,NaN
4,ANDA076224,PHARMOBEDIENT,ORIG,20030509,None,2003,FLUTAMIDE,CAPSULE,ORAL,Discontinued,FLUTAMIDE,NaN,NaN,NaN,NaN


In [18]:
df_orig[df_orig['active_ingredients']=="CLADRIBINE"]

,application_number,sponsor_name,submission_type,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,active_ingredients,openfda_brand_name,openfda_generic_name,openfda_route,openfda_substance_name
2062,NDA022561,EMD SERONO INC,ORIG,20190329,Type 3 - New Dosage Form,2019,MAVENCLAD,TABLET,ORAL,Prescription,CLADRIBINE,MAVENCLAD,CLADRIBINE,ORAL,CLADRIBINE
8585,ANDA200510,PHARMOBEDIENT,ORIG,20111006,Not Applicable,2011,CLADRIBINE,INJECTABLE,INJECTION,Discontinued,CLADRIBINE,NaN,NaN,NaN,NaN
11798,NDA020229,JANSSEN PHARMS,ORIG,19930226,Type 1 - New Molecular Entity,1993,LEUSTATIN,INJECTABLE,INJECTION,Discontinued,CLADRIBINE,NaN,NaN,NaN,NaN
12849,ANDA075405,HIKMA,ORIG,20000228,None,2000,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE
20606,ANDA076571,FRESENIUS KABI USA,ORIG,20040422,None,2004,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE
24464,ANDA210856,HISUN PHARM HANGZHOU,ORIG,20191125,None,2019,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE


## Label

In [39]:
file_nr = "01"
drug_label_file = f"/shares/animalwelfare.crs.uzh/fda_files/2025_11_11/drug-label-00{file_nr}-of-0013.json"

# Load your drug label JSON (replace filename with yours)
with open(drug_label_file, "r") as f:
    data = json.load(f)

results = data.get("results", [])

records = []

for result in tqdm(results, desc="Processing FDA drug labels", unit="label"):
    openfda = result.get("openfda", {})
    application_numbers = openfda.get("application_number", [])

    # ✅ Skip if no application_number
    if not application_numbers:
        continue

    clinical_text = " ".join(result.get("clinical_studies", []))
    nct_ids = re.findall(r"NCT\s*[-:]?\s*\d{6,8}", clinical_text, flags=re.IGNORECASE)

    # --- Extract only the first sentence from indications_and_usage
    indications_text = " ".join(result.get("indications_and_usage", []))
    match = re.match(r"^(.*?)(?<!\d)\.\s", indications_text)
    first_sentence = match.group(1).strip() if match else indications_text.strip()

    record = {
        "application_number": ", ".join(openfda.get("application_number", [])),
        "label_brand_name": ", ".join(openfda.get("brand_name", [])),
        "label_generic_name": ", ".join(openfda.get("generic_name", [])),
        "label_manufacturer_name": ", ".join(openfda.get("manufacturer_name", [])),
        "label_substance_name": ", ".join(openfda.get("substance_name", [])),
        "indications_first_sent": first_sentence,
        "indications_and_usage": indications_text,
        "clinical_studies": nct_ids,
    }

    records.append(record)

# Convert to DataFrame
df_labels = pd.DataFrame(records)
df_orig = df_orig.merge[df_labels, on="application_numer", how="left")

Processing FDA drug labels: 100%|██████████| 20000/20000 [00:00<00:00, 141800.54label/s]


In [41]:
import json
import re
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# --- only these columns will be written into df_orig
label_cols = [
    "label_brand_name",
    "label_generic_name",
    "label_manufacturer_name",
    "label_substance_name",
    "indications_first_sent",
    "indications_and_usage",
    "clinical_studies",
]

# ensure df_orig has target columns (so .update works cleanly)
for col in label_cols:
    if col not in df_orig.columns:
        df_orig[col] = pd.NA

# regex: first sentence without splitting on numbered list markers like "1."
FIRST_SENTENCE_RE = re.compile(r"^(.*?)(?<!\d)\.\s")

def first_sentence(text: str) -> str:
    if not text:
        return ""
    m = FIRST_SENTENCE_RE.match(text)
    return (m.group(1) if m else text).strip()

# --- iterate files 01..13, process and merge immediately
base_dir = Path("/shares/animalwelfare.crs.uzh/fda_files/2025_11_11")

for i in tqdm(range(1, 14), desc="Processing label files", unit="file"):
    file_nr = f"{i:02d}"
    drug_label_file = base_dir / f"drug-label-00{file_nr}-of-0013.json"

    with open(drug_label_file, "r") as f:
        data = json.load(f)

    results = data.get("results", [])

    # build minimal records for THIS file only
    records = []
    for result in tqdm(results, desc=f"Parsing {drug_label_file.name}", unit="label", leave=False):
        openfda = result.get("openfda", {}) or {}
        application_numbers = openfda.get("application_number", []) or []

        # Skip if no application_number
        if not application_numbers:
            continue

        # clinical trials: tolerant NCT regex (as you wrote)
        clinical_text = " ".join(result.get("clinical_studies", []) or [])
        nct_ids = re.findall(r"NCT\s*[-:]?\s*\d{6,8}", clinical_text, flags=re.IGNORECASE)

        # indications: first sentence only (don’t split on "1.")
        indications_text = " ".join(result.get("indications_and_usage", []) or [])
        match = re.match(r"^(.*?)(?<!\d)\.\s", indications_text)
        first_sent = match.group(1).strip() if match else indications_text.strip()

        # one row per application_number (clean merge key)
        for app_no in application_numbers:
            records.append({
                "application_number": app_no,
                "label_brand_name": ", ".join(openfda.get("brand_name", []) or []),
                "label_generic_name": ", ".join(openfda.get("generic_name", []) or []),
                "label_manufacturer_name": ", ".join(openfda.get("manufacturer_name", []) or []),
                "label_substance_name": ", ".join(openfda.get("substance_name", []) or []),
                "indications_first_sent": first_sent,
                "indications_and_usage": indications_text,
                "clinical_studies": nct_ids,  # keep as list
            })

    # turn this file's records into a df, dedupe by application_number to avoid row explosion
    df_labels_file = pd.DataFrame(records)
    if not df_labels_file.empty:
        df_labels_file = df_labels_file.drop_duplicates(subset=["application_number"], keep="first")
        # in-place update (no _x/_y columns, no row duplication)
        df_orig = df_orig.set_index("application_number")
        df_orig.update(df_labels_file.set_index("application_number")[label_cols])
        df_orig = df_orig.reset_index()

print("✅ Finished merging each label file into df_orig (on the fly).")


Processing label files: 100%|██████████| 13/13 [01:03<00:00,  4.90s/file]                        


✅ Finished merging each label file into df_orig (on the fly).


In [44]:
df_orig

,application_number,sponsor_name,submission_type,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
0,ANDA076194,WATSON LABS,ORIG,20020701,None,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,Prescription,...,LISINOPRIL AND HYDROCHLOROTHIAZIDE,ORAL,"HYDROCHLOROTHIAZIDE, LISINOPRIL",Lisinopril and Hydrochlorothiazide,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[]
1,ANDA076206,ROCKWELL MEDCL,ORIG,20030917,None,2003,CALCITRIOL,INJECTABLE,INJECTION,Discontinued,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,ANDA076212,APOTEX,ORIG,20040616,None,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,Discontinued,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,ANDA076215,FOUGERA PHARMS,ORIG,20031209,None,2003,BETAMETHASONE DIPROPIONATE,"CREAM, AUGMENTED",TOPICAL,Prescription,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,ANDA076224,PHARMOBEDIENT,ORIG,20030509,None,2003,FLUTAMIDE,CAPSULE,ORAL,Discontinued,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25707,ANDA076154,MYLAN,ORIG,20030820,None,2003,LORATADINE,TABLET,ORAL,Over-the-counter,...,LORATADINE,ORAL,LORATADINE,Loratadine,LORATADINE,Mylan Pharmaceuticals Inc.,LORATADINE,Uses temporarily relieves these symptoms due t...,Uses temporarily relieves these symptoms due t...,[]
25708,ANDA076155,APOTEX INC,ORIG,20030418,None,2003,IPRATROPIUM BROMIDE,"SPRAY, METERED",NASAL,Prescription,...,IPRATROPIUM BROMIDE,NASAL,IPRATROPIUM BROMIDE,Ipratropium Bromide,IPRATROPIUM BROMIDE,Apotex Corp.,IPRATROPIUM BROMIDE,INDICATIONS AND USAGE Ipratropium bromide nasa...,INDICATIONS AND USAGE Ipratropium bromide nasa...,[]
25709,ANDA076162,WATSON LABS TEVA,ORIG,20041014,None,2004,CARBOPLATIN,INJECTABLE,INJECTION,Discontinued,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
25710,ANDA076163,DR REDDYS,ORIG,20030905,None,2003,AMIODARONE HYDROCHLORIDE,INJECTABLE,INJECTION,Discontinued,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [43]:
df_orig[df_orig['active_ingredients']=="CLADRIBINE"]

,application_number,sponsor_name,submission_type,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
2062,NDA022561,EMD SERONO INC,ORIG,20190329,Type 3 - New Dosage Form,2019,MAVENCLAD,TABLET,ORAL,Prescription,...,CLADRIBINE,ORAL,CLADRIBINE,Mavenclad,CLADRIBINE,"EMD Serono, Inc.",CLADRIBINE,1 INDICATIONS AND USAGE MAVENCLAD is indicated...,1 INDICATIONS AND USAGE MAVENCLAD is indicated...,[NCT00213135]
8585,ANDA200510,PHARMOBEDIENT,ORIG,20111006,Not Applicable,2011,CLADRIBINE,INJECTABLE,INJECTION,Discontinued,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
11798,NDA020229,JANSSEN PHARMS,ORIG,19930226,Type 1 - New Molecular Entity,1993,LEUSTATIN,INJECTABLE,INJECTION,Discontinued,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
12849,ANDA075405,HIKMA,ORIG,20000228,None,2000,CLADRIBINE,INJECTABLE,INJECTION,Prescription,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,Hikma Pharmaceuticals USA Inc.,CLADRIBINE,"INDICATIONS AND USAGE Cladribine Injection, US...","INDICATIONS AND USAGE Cladribine Injection, US...",[]
20606,ANDA076571,FRESENIUS KABI USA,ORIG,20040422,None,2004,CLADRIBINE,INJECTABLE,INJECTION,Prescription,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,"Fresenius Kabi USA, LLC",CLADRIBINE,"INDICATIONS AND USAGE: Cladribine Injection, U...","INDICATIONS AND USAGE: Cladribine Injection, U...",[]
24464,ANDA210856,HISUN PHARM HANGZHOU,ORIG,20191125,None,2019,CLADRIBINE,INJECTABLE,INJECTION,Prescription,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,"Hisun Pharmaceuticals USA, Inc.",CLADRIBINE,"INDICATIONS FOR USE Cladribine Injection, USP ...","INDICATIONS FOR USE Cladribine Injection, USP ...",[]


In [45]:
df_orig.to_csv("out/FDA_drugs_labels_full.csv",index=False)